# Notebook 1: Generate Patch Attack Datasets

**VLM-ARB Cloud Framework**

This notebook generates and syncs clean + patch-poisoned datasets for patch attack evaluation.

## Workflow
1. Setup: Authenticate with Firebase and Google Drive
2. Prepare base dataset sources
3. Generate patch attack dataset (`patch_original` + `patch_poisoned`)
4. Upload to Cloud Storage
5. Log manifest to Firestore

**Key Feature**: Idempotent - if data already exists, generation can be skipped safely.

---

## Cell 1: Install Dependencies & Setup

In [4]:
# Install required pip packages (Colab environment)
import subprocess
import sys

packages = [
    'firebase-admin',
    'pillow',
    'numpy',
    'transformers',
    'torch',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All dependencies installed")

✅ All dependencies installed


In [ ]:
import sys
from pathlib import Path
import shutil
import json

# Mount Google Drive first
from google.colab import drive
drive.mount('/content/drive')

# Copy utilities folder from project to Colab workspace
project_dir = Path("/content/drive/MyDrive/VLM-ARB-Team")
source_utilities = project_dir / "utilities"
colab_utilities = Path("/root/utilities")
colab_utilities.mkdir(parents=True, exist_ok=True)

# If utilities not in Drive, create fallback implementations
if source_utilities.exists():
    print("📂 Found utilities in Drive, copying...")
    shutil.copytree(source_utilities, colab_utilities, dirs_exist_ok=True)
    print("✅ Utilities copied")
else:
    print("⚠️  Utilities folder not found in Drive")
    print("   Creating fallback implementations for cloud sync...")
    (colab_utilities / "__init__.py").write_text("# Utilities module\n")

    fallback_cloud_sync = '''
from pathlib import Path
from typing import Dict, Any
import firebase_admin
from firebase_admin import credentials, firestore, storage

class FirebaseSync:
    def __init__(self, credentials_path: str, bucket_name: str = None):
        self.offline_mode = False
        self.db = None
        self.bucket = None
        try:
            cred = credentials.Certificate(credentials_path)
            project_id = cred.project_id
            resolved_bucket = bucket_name or f"{project_id}.appspot.com"
            if not firebase_admin._apps:
                firebase_admin.initialize_app(cred, {"storageBucket": resolved_bucket})
            self.db = firestore.client()
            self.bucket = storage.bucket()
        except Exception as e:
            print(f"⚠️ Firebase init fallback failed: {e}")
            self.offline_mode = True

    def upload_file(self, local_path: str, bucket_path: str, make_public: bool = False) -> bool:
        if self.offline_mode or not self.bucket:
            return False
        p = Path(local_path)
        if not p.exists():
            return False
        try:
            blob = self.bucket.blob(bucket_path)
            blob.upload_from_filename(str(p))
            if make_public:
                blob.make_public()
            return True
        except Exception as e:
            print(f"Upload failed for {local_path}: {e}")
            return False

    def upload_image_batch(self, image_dir: str, bucket_path_prefix: str):
        uploaded = {}
        p = Path(image_dir)
        if not p.exists():
            return uploaded
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
            for img in p.glob(ext):
                bucket_path = f"{bucket_path_prefix}{img.name}"
                if self.upload_file(str(img), bucket_path, make_public=False):
                    uploaded[img.name] = bucket_path
        return uploaded

    def upload_results(self, run_id: str, metrics_dict: Dict[str, Any], metadata: Dict[str, Any] = None, collection: str = "results") -> bool:
        if self.offline_mode or not self.db:
            return False
        payload = {"metrics": metrics_dict}
        if metadata:
            payload.update(metadata)
        try:
            self.db.collection(collection).document(run_id).set(payload, merge=True)
            return True
        except Exception as e:
            print(f"Firestore upload failed: {e}")
            return False
'''
    (colab_utilities / "cloud_sync.py").write_text(fallback_cloud_sync)
    print("✅ Fallback utilities.cloud_sync created")

# Add to path
if '/root' not in sys.path:
    sys.path.insert(0, '/root')

# Authenticate with Google Drive
team_folder = Path("/content/drive/MyDrive/VLM-ARB-Team")
print("\n📊 Environment Ready:")
print(f"  Team Folder: {team_folder}")
print("  Google Drive: ✅ Mounted")

# Initialize Firebase
fs = None
creds_path = team_folder / "secrets" / "serviceAccountKey.json"

if creds_path.exists():
    print(f"✅ Found credentials at: {creds_path}")
    try:
        from utilities.cloud_sync import FirebaseSync
        fs = FirebaseSync(str(creds_path))
        if fs and not fs.offline_mode:
            print("✅ Firebase authenticated - uploads enabled")
        else:
            print("⚠️ Firebase object created, but running offline")
    except Exception as e:
        print(f"❌ Firebase initialization failed: {str(e)[:200]}")
        print("   Will use Google Drive only")
        fs = None
else:
    print("\n⚠️  Firebase credentials NOT found")
    print(f"   Expected at: {creds_path}")
    print("\n📋 To enable Firebase, upload your credentials:")
    print("   1. Get serviceAccountKey.json from Firebase Console")
    print("   2. Upload to: Drive/VLM-ARB-Team/secrets/serviceAccountKey.json")
    print("   3. Re-run this cell")
    print("\n   For now, continuing with Google Drive only...")

# Prepare context
context = {
    'user_email': 'colab_user',
    'creds_path': creds_path,
    'firebase_available': fs is not None and not getattr(fs, 'offline_mode', True)
}

# Check idempotency
drive_datasets_dir = team_folder / "datasets"
skip_generation = False

if drive_datasets_dir.exists() and list(drive_datasets_dir.glob("*")):
    print("\n✅ Dataset already exists on Drive")
    skip_generation = True
    print("⏭️  Skipping generation (idempotent)")
else:
    print("\n🆕 No existing dataset - will generate new data")

MessageError: User cancelled dfs_ephemeral authorization

## Cell 3: Define Dataset Generation Functions

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import json
from datetime import datetime

def generate_synthetic_image(width=256, height=256, seed=None):
    """
    Generate a random synthetic image for testing.
    
    Args:
        width, height: Image dimensions
        seed: Random seed for reproducibility
    
    Returns:
        PIL Image object
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Create image with random colors
    img_array = np.random.randint(0, 256, (height, width, 3), dtype=np.uint8)
    
    # Add some structure: shapes
    img = Image.fromarray(img_array)
    draw = ImageDraw.Draw(img)
    
    # Draw random circles and rectangles
    for _ in range(5):
        x0 = np.random.randint(0, width)
        y0 = np.random.randint(0, height)
        x1 = np.random.randint(x0, width)
        y1 = np.random.randint(y0, height)
        color = tuple(np.random.randint(0, 256, 3))
        draw.rectangle([x0, y0, x1, y1], fill=color)
    
    # Draw some circles
    for _ in range(3):
        x = np.random.randint(50, width-50)
        y = np.random.randint(50, height-50)
        r = np.random.randint(10, 50)
        color = tuple(np.random.randint(0, 256, 3))
        draw.ellipse([x-r, y-r, x+r, y+r], fill=color)
    
    return img

print("✅ Dataset generation functions defined")

## Cell 3: Define Dataset Generation Functions

In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import hashlib
import json
from datetime import datetime

def generate_synthetic_image(width=256, height=256, seed=None):
    """
    Generate a random synthetic image for testing.
    
    Args:
        width, height: Image dimensions
        seed: Random seed for reproducibility
    
    Returns:
        PIL Image object
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Create image with random colors
    img_array = np.random.randint(0, 256, (height, width, 3), dtype=np.uint8)
    
    # Add some structure: shapes
    img = Image.fromarray(img_array)
    draw = ImageDraw.Draw(img)
    
    # Draw random circles and rectangles
    for _ in range(5):
        x0 = np.random.randint(0, width)
        y0 = np.random.randint(0, height)
        x1 = np.random.randint(x0, width)
        y1 = np.random.randint(y0, height)
        color = tuple(np.random.randint(0, 256, 3))
        draw.rectangle([x0, y0, x1, y1], fill=color)
    
    # Draw some circles
    for _ in range(3):
        x = np.random.randint(50, width-50)
        y = np.random.randint(50, height-50)
        r = np.random.randint(10, 50)
        color = tuple(np.random.randint(0, 256, 3))
        draw.ellipse([x-r, y-r, x+r, y+r], fill=color)
    
    return img

print("✅ Dataset generation functions defined")

## Cell 4: Generate Base Synthetic Images

In [ ]:
from pathlib import Path

if not skip_generation:
    # Create temporary directory for images
    test_images_dir = Path("/tmp/vlm_arb_test_images")
    test_images_dir.mkdir(exist_ok=True)
    
    print(f"🖼️  Generating base images...")
    
    # Generate 5 base synthetic images
    base_images = {}
    num_images = 5
    
    for i in range(num_images):
        img = generate_synthetic_image(seed=42 + i)  # Reproducible
        img_path = test_images_dir / f"base_image_{i:03d}.png"
        img.save(img_path)
        base_images[f"base_image_{i:03d}"] = str(img_path)
        print(f"   ✓ Created: {img_path.name}")
    
    print(f"✅ Generated {num_images} base images")
    
else:
    print("⏭️  Skipping base image generation (using existing dataset)")
    base_images = {}

## Cell 5: Generate Patch Attack Dataset

In [ ]:
import subprocess
import sys
from pathlib import Path

if not skip_generation:
    print("🧩 Generating patch attack dataset...")
    
    patch_input_dir = project_dir / "datasets" / "Pertubation_original"
    patch_clean_dir = project_dir / "datasets" / "patch_original"
    patch_poison_dir = project_dir / "datasets" / "patch_poisoned"
    patch_labels = patch_poison_dir / "labels.csv"
    
    script_path = project_dir / "scripts" / "generate_patch_dataset.py"
    if not script_path.exists():
        raise FileNotFoundError(f"Patch generator script not found: {script_path}")
    
    cmd = [
        sys.executable,
        str(script_path),
        "--input-dir", str(patch_input_dir),
        "--clean-output-dir", str(patch_clean_dir),
        "--poison-output-dir", str(patch_poison_dir),
        "--mapping-csv", str(patch_labels),
        "--patch-size", "64",
        "--opacity", "0.95",
        "--random-seed", "42",
    ]
    
    print("   Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(project_dir))
    
    patch_clean_count = len(list((patch_clean_dir / "images").glob("*.jpg"))) + len(list((patch_clean_dir / "images").glob("*.png")))
    patch_poison_count = len(list((patch_poison_dir / "images").glob("*.jpg"))) + len(list((patch_poison_dir / "images").glob("*.png")))
    variant_count = patch_poison_count
    
    print("✅ Patch dataset generated")
    print(f"   Clean images: {patch_clean_count}")
    print(f"   Poisoned images: {patch_poison_count}")
    print(f"   Labels: {patch_labels}")
else:
    print("⏭️  Skipping patch generation")
    patch_clean_dir = project_dir / "datasets" / "patch_original"
    patch_poison_dir = project_dir / "datasets" / "patch_poisoned"
    patch_labels = patch_poison_dir / "labels.csv"
    patch_clean_count = len(list((patch_clean_dir / "images").glob("*.jpg"))) if (patch_clean_dir / "images").exists() else 0
    patch_poison_count = len(list((patch_poison_dir / "images").glob("*.jpg"))) if (patch_poison_dir / "images").exists() else 0
    variant_count = patch_poison_count

## Cell 6: Verify Patch Dataset on Google Drive

In [ ]:
from pathlib import Path

print("💾 Verifying patch datasets on Google Drive...")

drive_datasets_dir = team_folder / "datasets"
drive_patch_clean_dir = drive_datasets_dir / "patch_original" / "images"
drive_patch_poison_dir = drive_datasets_dir / "patch_poisoned" / "images"
drive_patch_labels = drive_datasets_dir / "patch_poisoned" / "labels.csv"

drive_patch_clean_dir.mkdir(parents=True, exist_ok=True)
drive_patch_poison_dir.mkdir(parents=True, exist_ok=True)

# Source is already generated in project_dir (Drive-backed)
patch_clean_src = project_dir / "datasets" / "patch_original" / "images"
patch_poison_src = project_dir / "datasets" / "patch_poisoned" / "images"
patch_labels_src = project_dir / "datasets" / "patch_poisoned" / "labels.csv"

clean_count = len(list(patch_clean_src.glob("*.jpg"))) + len(list(patch_clean_src.glob("*.png"))) if patch_clean_src.exists() else 0
poison_count = len(list(patch_poison_src.glob("*.jpg"))) + len(list(patch_poison_src.glob("*.png"))) if patch_poison_src.exists() else 0
labels_ok = patch_labels_src.exists()

print("✅ Patch dataset check complete")
print(f"   Clean images: {clean_count}")
print(f"   Poisoned images: {poison_count}")
print(f"   Labels present: {labels_ok}")

patch_clean_dir = project_dir / "datasets" / "patch_original"
patch_poison_dir = project_dir / "datasets" / "patch_poisoned"
patch_labels = patch_labels_src

## Cell 7: Upload Patch Clean Images to Cloud Storage

In [ ]:
if fs and not fs.offline_mode:
    print("☁️ Uploading patch clean images to Cloud Storage...")
    
    patch_clean_images = patch_clean_dir / "images"
    uploaded_clean = 0
    if patch_clean_images.exists():
        for p in sorted(patch_clean_images.glob("*")):
            if p.is_file() and p.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp"]:
                bucket_path = f"datasets/patch_original/images/{p.name}"
                if fs.upload_file(str(p), bucket_path, make_public=False):
                    uploaded_clean += 1
    
    print(f"✅ Uploaded patch clean images: {uploaded_clean}")
else:
    print("⏭️ Skipping upload (offline mode or Firebase unavailable)")

## Cell 8: Upload Patch Poisoned Images + Labels

In [ ]:
if fs and not fs.offline_mode:
    print("☁️ Uploading patch poisoned images and labels...")
    
    uploaded_total = 0
    patch_poison_images = patch_poison_dir / "images"
    if patch_poison_images.exists():
        urls_poison = fs.upload_image_batch(str(patch_poison_images), "datasets/patch_poisoned/images/")
        uploaded_total += len(urls_poison)
        print(f"   ✓ Poisoned images uploaded: {len(urls_poison)}")
    
    if patch_labels.exists():
        labels_ok = fs.upload_file(
            str(patch_labels),
            "datasets/patch_poisoned/labels.csv",
            make_public=False
        )
        if labels_ok:
            print("   ✓ labels.csv uploaded")
    
    print(f"✅ Total patch poisoned uploads: {uploaded_total}")
else:
    print("⏭️ Skipping patch upload (offline mode or Firebase unavailable)")

## Cell 9: Log Patch Dataset Manifest to Firestore

In [ ]:
import subprocess
from datetime import datetime

try:
    git_hash = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'],
        cwd='/root'
    ).decode().strip()[:8]
except Exception:
    git_hash = "unknown"

dataset_version = f"v{datetime.now().strftime('%Y%m%d_%H%M%S')}_{git_hash}"

patch_clean_images = patch_clean_dir / "images"
patch_poison_images = patch_poison_dir / "images"
clean_count = len(list(patch_clean_images.glob("*.jpg"))) + len(list(patch_clean_images.glob("*.png"))) if patch_clean_images.exists() else 0
poison_count = len(list(patch_poison_images.glob("*.jpg"))) + len(list(patch_poison_images.glob("*.png"))) if patch_poison_images.exists() else 0

dataset_manifest = {
    "dataset_version": dataset_version,
    "dataset_info": {
        "base_image_count": clean_count,
        "total_variants": poison_count,
        "attack_types": ["patch"],
        "created_at": datetime.now().isoformat(),
        "created_by": context['user_email'],
        "git_version": git_hash,
        "storage_paths": {
            "patch_clean": "/content/drive/MyDrive/VLM-ARB-Team/datasets/patch_original/images",
            "patch_poisoned": "/content/drive/MyDrive/VLM-ARB-Team/datasets/patch_poisoned/images",
            "patch_labels": "/content/drive/MyDrive/VLM-ARB-Team/datasets/patch_poisoned/labels.csv"
        },
        "patch_config": {
            "patch_size": 64,
            "opacity": 0.95,
            "random_seed": 42
        }
    }
}

if fs and context.get('firebase_available'):
    print("☁️ Logging patch dataset manifest to Firestore...")
    try:
        fs.upload_results(
            run_id="patch_dataset_current",
            metrics_dict=dataset_manifest,
            metadata={"status": "active", "type": "dataset_patch"},
            collection="team_configs"
        )
        print("✅ Patch dataset manifest uploaded to Firestore")
    except Exception as e:
        print(f"⚠️ Firestore upload failed: {str(e)[:150]}")
        print("   Data is safe on Google Drive")
else:
    print("💾 Firebase not available - patch dataset saved to Google Drive only")

print("\n📊 Patch Dataset Summary:")
print(f"   Version: {dataset_version}")
print(f"   Clean Images: {clean_count}")
print(f"   Poisoned Images: {poison_count}")
print(f"   Labels: {patch_labels}")
print("   Storage: Google Drive ✅")
if context.get('firebase_available'):
    print("   Firestore: ✅ Logged")
else:
    print("   Firestore: ⚠️ Not configured")

## Cell 10: Verify & Summary

In [ ]:
print("\n" + "="*60)
print("✅ PATCH DATASET GENERATION COMPLETE")
print("="*60)

patch_clean_images = patch_clean_dir / "images"
patch_poison_images = patch_poison_dir / "images"
clean_count = len(list(patch_clean_images.glob("*.jpg"))) + len(list(patch_clean_images.glob("*.png"))) if patch_clean_images.exists() else 0
poison_count = len(list(patch_poison_images.glob("*.jpg"))) + len(list(patch_poison_images.glob("*.png"))) if patch_poison_images.exists() else 0

print(f"\n📦 Patch Dataset Ready")
print(f"   Clean Images: {clean_count}")
print(f"   Poisoned Images: {poison_count}")
print(f"   Labels File: {patch_labels}")
print(f"   Clean Path: {patch_clean_dir}")
print(f"   Poison Path: {patch_poison_dir}")
print(f"   Storage: Google Drive ✅")
if context.get('firebase_available'):
    print("   Firestore: ✅ Logged")
else:
    print("   Firestore: ⚠️ Not configured")

print("\n📋 Next Steps:")
print("   1. Open Notebook 2 for evaluation")
print("   2. Use mode=patch with clean=patch_original and poisoned=patch_poisoned")
print("   3. Use datasets/patch_poisoned/labels.csv for pair mapping")
print("="*60)